In [2]:
!pip install alpha_vantage

In [3]:
import pandas as pd
import requests
import json
from alpha_vantage.timeseries import TimeSeries

api_key = 'DGLLCB818CWYTWLU'
symbol = 'AAPL'
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol={symbol}&interval=5min&apikey={api_key}'
r = requests.get(url)

In [4]:
data = r.json()['Time Series (5min)']

In [5]:
#Importando os dados temporais mensais para um Data Frame
df = data
df = pd.DataFrame(data).T.reset_index()
#Renomiando as colunas e organizando os tipos de dados das colunas
df = pd.DataFrame.from_dict(data, orient='index').reset_index()
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
df = df.astype({'Date': 'datetime64[ns]', 'Open': 'float', 'High': 'float', 'Low':'float', 'Close': 'float', 'Volume': 'float'})
df

,Date,Open,High,Low,Close,Volume
0,2025-05-02 19:55:00,205.1300,205.150,205.04,205.1099,16536.0
1,2025-05-02 19:50:00,205.1100,205.130,205.09,205.1200,4281.0
2,2025-05-02 19:45:00,205.1100,205.110,205.04,205.1100,4751.0
3,2025-05-02 19:40:00,205.1100,205.120,205.05,205.1000,2894.0
4,2025-05-02 19:35:00,205.1199,205.130,205.10,205.1200,6896.0
...,...,...,...,...,...,...
95,2025-05-02 12:00:00,205.4874,206.420,205.30,206.3700,1345740.0
96,2025-05-02 11:55:00,205.0600,205.580,204.98,205.4800,470233.0
97,2025-05-02 11:50:00,205.4500,205.535,205.00,205.0600,483688.0
98,2025-05-02 11:45:00,205.5700,205.640,205.25,205.4500,742456.0


In [13]:
# Lista de colunas numéricas
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

# Aplicando o método IQR em todas as colunas
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Mantendo apenas os valores dentro dos limites
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [14]:
#Normalização dos dados
cols_to_normalize = ['Open', 'High', 'Low', 'Close', 'Volume']
df[cols_to_normalize] = (df[cols_to_normalize] - df[cols_to_normalize].mean()) / df[cols_to_normalize].std()


In [ ]:
#Exportar em .parquet
df.to_parquet('timeseries_monthly_APPLE.parquet', engine='pyarrow')